# The Price Is Right

### And now, to evaluate our fine-tuned open source model



In [ ]:
!pip install -q --upgrade bitsandbytes trl
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel
from util import evaluate

In [ ]:
# Constants

PROJECT_NAME = "price"
HF_USER = "meayesha24"

MODELS_TO_TEST = [

    {
        "name": "llama-1000",
        "base": "meta-llama/Llama-3.2-3B",
        "adapter": "meayesha24/price-llama-1000-2026-03-07_18.55.04-lite"
    },

    {
        "name": "llama-5000",
        "base": "meta-llama/Llama-3.2-3B",
        "adapter": "meayesha24/price-llama-5000-2026-03-07_19.37.14-lite"
    },

    {
        "name": "mistral-5000",
        "base": "mistralai/Mistral-7B-Instruct-v0.3",
        "adapter": "meayesha24/price-mistral-5000-2026-03-07_19.45.02-lite"
    }

]

LITE_MODE = True

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

if LITE_MODE:
  RUN_NAME = "2025-11-30_15.10.55-lite"
  REVISION = None
else:
  RUN_NAME = "2025-11-28_18.47.07"
  REVISION = "b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hyper-parameters - QLoRA

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

### Log in to HuggingFace

If you don't already have a HuggingFace account, visit https://huggingface.co to sign up and create a token.

Then select the Secrets for this Notebook by clicking on the key icon in the left, and add a new secret called `HF_TOKEN` with the value as your token.


In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HG-Token')
login(hf_token, add_to_git_credential=True)

In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

In [ ]:
test[0]

## Now load the Tokenizer and Model

In [ ]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [ ]:
# @title
results = {}

for config in MODELS_TO_TEST:

    print(f"\nLoading {config['name']}")

    tokenizer = AutoTokenizer.from_pretrained(
        config["base"],
        trust_remote_code=True
    )

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        config["base"],
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True
    )

    base_model.generation_config.pad_token_id = tokenizer.pad_token_id

    fine_tuned_model = PeftModel.from_pretrained(
        base_model,
        config["adapter"]
    )

    print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

    def model_predict(item):
        inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")

        with torch.no_grad():
            output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)

        prompt_len = inputs["input_ids"].shape[1]
        generated_ids = output_ids[0, prompt_len:]

        return tokenizer.decode(generated_ids)

    set_seed(42)

    score = evaluate(model_predict, test)

    results[config["name"]] = score

    print(f"{config['name']} MAE: {score}")

In [ ]:
print("\nFinal Results")

for name, score in results.items():
    print(f"{name}: {score}")